# Analysis Metrics Dashboard

用于查看 `db/analysis_metrics.db` 的看板与各板块数据。

使用方法：
1. 运行第 1、2 个代码单元（连接 + 参数）
2. 修改 `TARGET_DATE` / `TARGET_SOURCE`
3. 依次运行后续查询单元

In [24]:
import sqlite3
import pandas as pd
from IPython.display import display

DB_METRICS = '/Users/yy/.openclaw/workspace/db/analysis_metrics.db'
conn = sqlite3.connect(DB_METRICS)
print('Connected:', DB_METRICS)

Connected: /Users/yy/.openclaw/workspace/db/analysis_metrics.db


In [25]:
# 参数：按需修改
TARGET_DATE = '2026-04-01'  # 设为 None 表示看全部日期
TARGET_SOURCE = None        # 'k1' / 'k2' / 'k12'；设为 None 表示看全部

where = []
if TARGET_DATE:
    where.append(f"日期 = '{TARGET_DATE}'")
if TARGET_SOURCE:
    where.append(f"数据源 = '{TARGET_SOURCE}'")
WHERE_SQL = ('WHERE ' + ' AND '.join(where)) if where else ''
print('WHERE_SQL =', WHERE_SQL)

WHERE_SQL = WHERE 日期 = '2026-04-01'


In [26]:
# 1) 看板总览（中文视图）
sql_dashboard = f"""
SELECT 日期, 数据源, 行数, 理论利润, 总费用, alpha, slippage, 今日总盈亏,
       校验项总数, 校验失败数, 失败率, 最大绝对差, 更新时间
FROM 看板_每日快照
{WHERE_SQL}
ORDER BY 日期 DESC, 数据源
"""
df_dashboard = pd.read_sql_query(sql_dashboard, conn)
display(df_dashboard)

,日期,数据源,行数,理论利润,总费用,alpha,slippage,今日总盈亏,校验项总数,校验失败数,失败率,最大绝对差,更新时间
0,2026-04-01,k1,42816,20958.43573,5679.505474,-15468.216830,-9788.711355,11498.560173,8,8,1.000,5749.280173,2026-04-03 03:06:05
1,2026-04-01,k12,98036,57096.58729,10124.491412,-46449.592232,-36325.100820,11165.174033,8,7,0.875,0.004033,2026-04-03 03:06:07
2,2026-04-01,k2,55221,36138.15156,4444.985937,-30981.375402,-26536.389464,5415.893947,8,7,0.875,0.004598,2026-04-03 03:06:06


In [27]:
# 2) 买卖方向（中文视图）
sql_direction = f"""
SELECT 日期, 数据源, 方向, 笔数, alpha累计, slippage累计, 今日总盈亏累计, alpha均值, slippage均值
FROM 看板_买卖方向
{WHERE_SQL}
ORDER BY 日期 DESC, 数据源
"""
df_direction = pd.read_sql_query(sql_direction, conn)
display(df_direction)

,日期,数据源,方向,笔数,alpha累计,slippage累计,今日总盈亏累计,alpha均值,slippage均值
0,2026-04-01,k1,buy,21201,52276.2496,55352.3886,0.0,2.4657,2.6108
1,2026-04-01,k1,sell,21615,-67744.4665,-65141.0999,0.0,-3.1343,-3.0138
2,2026-04-01,k12,buy,48688,95674.3193,101038.6291,0.0,1.9650,2.0752
3,2026-04-01,k12,sell,49348,-142123.9115,-137363.7299,0.0,-2.8800,-2.7836
4,2026-04-01,k2,buy,27487,43398.0697,45686.2405,0.0,1.5789,1.6621
5,2026-04-01,k2,sell,27734,-74379.4451,-72222.6300,0.0,-2.6819,-2.6041


In [28]:
# 3) 规模区间（中文视图）
sql_size = f"""
SELECT 日期, 数据源, 规模区间, 笔数, alpha累计, slippage累计, 今日总盈亏累计,
       alpha均值, slippage均值, 理论利润累计
FROM 看板_规模区间
{WHERE_SQL}
ORDER BY 日期 DESC, 数据源, 规模区间
"""
df_size = pd.read_sql_query(sql_size, conn)
display(df_size)

,日期,数据源,规模区间,笔数,alpha累计,slippage累计,今日总盈亏累计,alpha均值,slippage均值,理论利润累计
0,2026-04-01,k1,100-500,20655,-1996.3179,-786.2849,1928.091776,-0.0967,-0.0381,3800.660872
1,2026-04-01,k1,1k-5k,5971,-8923.3323,-6067.3126,3061.753109,-1.4944,-1.0161,11974.143755
2,2026-04-01,k1,50-100,4897,-27.8774,27.6148,251.888867,-0.0057,0.0056,210.497849
3,2026-04-01,k1,500-1k,8909,-3319.3652,-2073.5270,703.088190,-0.3726,-0.2327,4028.198874
4,2026-04-01,k1,<50,2319,-550.2758,-530.7451,-354.673987,-0.2373,-0.2289,73.175002
5,2026-04-01,k1,>5k,59,-644.0194,-353.0543,160.758756,-10.9156,-5.9840,866.357144
6,2026-04-01,k12,100-500,43080,-5477.5318,-3260.7539,2050.721278,-0.1271,-0.0757,7382.835244
7,2026-04-01,k12,1k-5k,10527,-13130.4768,-8601.4678,7810.032405,-1.2473,-0.8171,20799.560523
8,2026-04-01,k12,50-100,13004,-357.7889,-220.3937,72.713568,-0.0275,-0.0169,478.828375
9,2026-04-01,k12,500-1k,26455,-8791.7776,-5979.9028,2432.715994,-0.3323,-0.2260,11016.940853


In [29]:
# 4) 价差区间 + 池子（中文视图）
sql_spread = f"""
SELECT 日期, 数据源, 价差区间, 笔数, alpha累计, slippage累计,
       理论利润累计, 捆绑费累计, 今日总盈亏累计
FROM 看板_价差区间
{WHERE_SQL}
ORDER BY 日期 DESC, 数据源, 价差区间
"""

sql_pool = f"""
SELECT 日期, 数据源, 池子类型, 笔数, alpha累计, slippage累计,
       理论利润累计, 捆绑费累计, gas费累计, 今日总盈亏累计
FROM 看板_池子类型
{WHERE_SQL}
ORDER BY 日期 DESC, 数据源, 池子类型
"""

df_spread = pd.read_sql_query(sql_spread, conn)
df_pool = pd.read_sql_query(sql_pool, conn)
print('价差区间')
display(df_spread)
print('池子类型')
display(df_pool)

价差区间


,日期,数据源,价差区间,笔数,alpha累计,slippage累计,理论利润累计,捆绑费累计,今日总盈亏累计
0,2026-04-01,k1,0.05-0.10%,14212,-1166.919944,-425.922560,2174.659513,664.931921,1091.004399
1,2026-04-01,k1,0.10-0.15%,12533,-3865.160752,-2471.957529,4504.784760,1326.975319,662.231679
2,2026-04-01,k1,0.15-0.20%,4124,-1679.522417,-868.774969,2561.604286,788.957498,1071.578951
3,2026-04-01,k1,0.20-0.50%,4338,-3427.286426,-1643.635405,5614.213187,1760.786273,2115.197471
4,2026-04-01,k1,<0.05%,4390,-1421.886424,-1292.340212,488.479752,106.047618,-909.037063
5,2026-04-01,k1,>0.50%,1471,-10183.992421,-9422.466326,2823.399890,753.644084,-7340.131315
6,2026-04-01,k12,0.05-0.10%,37532,-5811.072253,-4320.202851,5420.189016,1291.780590,-307.850227
7,2026-04-01,k12,0.10-0.15%,23771,-5080.224606,-2900.716297,8183.043563,2054.482078,3300.791295
8,2026-04-01,k12,0.15-0.20%,9103,-4643.316518,-3288.125419,5247.399606,1307.355165,807.196873
9,2026-04-01,k12,0.20-0.50%,8972,-6502.264606,-3466.950708,11041.952962,2987.999396,4589.775091


池子类型


,日期,数据源,池子类型,笔数,alpha累计,slippage累计,理论利润累计,捆绑费累计,gas费累计,今日总盈亏累计
0,2026-04-01,k1,v2,29,-75.675400,-74.432568,-21.521900,1.111741,0.131090,-98.095292
1,2026-04-01,k1,v3,12813,-7238.022732,-4827.174406,9451.230156,2346.076486,64.771840,2243.911727
2,2026-04-01,k1,v4,29973,-8154.518697,-4887.104381,11528.727474,3104.463184,162.951133,3603.463652
3,2026-04-01,k12,v2,120,-71.762962,-8.068437,-28.361848,63.147075,0.547451,-60.609130
4,2026-04-01,k12,v3,31374,-28627.811579,-24365.165518,32251.106692,4104.821116,157.824946,3738.195221
5,2026-04-01,k12,v4,66542,-17750.017690,-11951.866865,24873.842446,5436.842333,361.308492,7487.587942
6,2026-04-01,k2,v2,91,3.912438,66.364131,-6.839948,62.035333,0.416360,37.486162
7,2026-04-01,k2,v3,18561,-21389.788847,-19537.991111,22799.876536,1758.744630,93.053106,1494.283494
8,2026-04-01,k2,v4,36569,-9595.498992,-7064.762484,13345.114972,2332.379148,198.357360,3884.124290


In [30]:
# 5) Token 与捆绑费 Token（中文视图）
sql_token = f"""
SELECT 日期, 数据源, Token, 笔数, 今日总盈亏累计, alpha累计, slippage累计,
       理论利润累计, 捆绑费累计, gas费累计, 买币规模, 卖币规模, 净买卖规模
FROM 看板_Token
{WHERE_SQL}
ORDER BY 日期 DESC, 数据源, 今日总盈亏累计 DESC
LIMIT 200
"""

sql_bundle = f"""
SELECT 日期, 数据源, Token, 笔数, 捆绑费累计, 理论利润累计,
       捆绑费率百分比, alpha累计, 今日总盈亏累计
FROM 看板_捆绑费Token
{WHERE_SQL}
ORDER BY 日期 DESC, 数据源, 捆绑费率百分比 DESC
LIMIT 200
"""

df_token = pd.read_sql_query(sql_token, conn)
df_bundle = pd.read_sql_query(sql_bundle, conn)
print('Token汇总')
display(df_token)
print('捆绑费Token')
display(df_bundle)

Token汇总


,日期,数据源,Token,笔数,今日总盈亏累计,alpha累计,slippage累计,理论利润累计,捆绑费累计,gas费累计,买币规模,卖币规模,净买卖规模
0,2026-04-01,k1,UNKNOWN,0,5749.280086,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000
1,2026-04-01,k1,DAO,791,1309.810009,1079.025438,1213.496248,244.728546,130.525758,3.945052,7.098707e+04,7.025546e+04,731.613779
2,2026-04-01,k1,EDGE,2691,1159.565526,-729.768839,-485.125906,1896.803262,230.126951,14.515981,1.320190e+06,1.320816e+06,-625.657886
3,2026-04-01,k1,BASED,2123,907.268768,-1606.539825,-1216.209726,2510.795479,378.855144,11.474955,1.081373e+06,1.081991e+06,-618.248430
4,2026-04-01,k1,STABLE,1731,549.461314,-602.300586,-294.547369,1171.715167,299.046800,8.706417,7.239269e+05,7.253491e+05,-1422.153695
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,2026-04-01,k12,ROBO,292,-185.007468,-127.587086,-24.375315,141.729097,101.525623,1.686147,2.163007e+05,2.164180e+05,-117.353782
196,2026-04-01,k12,IR,589,-186.222359,-312.050593,-282.651709,166.049589,26.346504,3.052381,6.924569e+04,6.916931e+04,76.380765
197,2026-04-01,k12,RTX,779,-191.349757,-866.906326,-816.741175,697.151767,45.810579,4.354573,3.334109e+05,3.344291e+05,-1018.180093
198,2026-04-01,k12,Q,635,-209.486166,-379.424617,-338.385868,132.198082,37.952769,3.085980,8.987154e+04,9.092158e+04,-1050.040199


捆绑费Token


,日期,数据源,Token,笔数,捆绑费累计,理论利润累计,捆绑费率百分比,alpha累计,今日总盈亏累计
0,2026-04-01,k1,$BANANA,293,1.782740e+02,180.274825,9.889011e+01,120.154000,463.840372
1,2026-04-01,k1,XPL,104,2.537160e+01,29.612645,8.567827e+01,56.130683,101.275655
2,2026-04-01,k1,C,227,6.833230e+01,81.408069,8.393799e+01,-75.244565,0.735606
3,2026-04-01,k1,KITE,1099,2.916894e+02,371.216553,7.857661e+01,-374.547131,-85.384741
4,2026-04-01,k1,Beat,1302,2.430047e+02,315.090735,7.712215e+01,-398.106312,-70.174580
...,...,...,...,...,...,...,...,...,...
195,2026-04-01,k12,KAVA,3,2.316266e-02,0.664378,3.486368e+00,1.991330,2.655708
196,2026-04-01,k12,BSU,11,6.770000e-13,0.205893,3.288116e-10,-0.258811,5.149691
197,2026-04-01,k12,USDT,3,0.000000e+00,17999.356961,0.000000e+00,-18311.403049,-314.425094
198,2026-04-01,k12,老子,34,3.231005e+00,-3.671953,-8.799146e+01,5.012565,18.031237


In [31]:
# 6) 报告校验异常（中文视图）
sql_reconcile = f"""
SELECT 日期, 数据源, 指标名, 报告值, 重算值, 绝对差, 状态, 报告路径, 创建时间
FROM 看板_校验差异
{WHERE_SQL}
ORDER BY 日期 DESC, 数据源, 指标名
"""

df_reconcile = pd.read_sql_query(sql_reconcile, conn)
display(df_reconcile)
print('仅失败项:')
display(df_reconcile[df_reconcile['状态'] == 'fail'])

,日期,数据源,指标名,报告值,重算值,绝对差,状态,报告路径,创建时间
0,2026-04-01,k1,rows_count,42815.00,42816.000000,1.000000,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 02:51:27
1,2026-04-01,k1,rows_count,42815.00,42816.000000,1.000000,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 03:06:05
2,2026-04-01,k1,total_alpha,-15468.22,-15468.216830,0.003170,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 02:51:27
3,2026-04-01,k1,total_alpha,-15468.22,-15468.216830,0.003170,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 03:06:05
4,2026-04-01,k1,total_bundle_fee,5451.65,5451.651412,0.001412,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 02:51:27
5,2026-04-01,k1,total_bundle_fee,5451.65,5451.651412,0.001412,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 03:06:05
6,2026-04-01,k1,total_fee,5679.50,5679.505474,0.005474,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 02:51:27
7,2026-04-01,k1,total_fee,5679.50,5679.505474,0.005474,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 03:06:05
8,2026-04-01,k1,total_gas_fee,227.85,227.854062,0.004062,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 02:51:27
9,2026-04-01,k1,total_gas_fee,227.85,227.854062,0.004062,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 03:06:05


仅失败项:


,日期,数据源,指标名,报告值,重算值,绝对差,状态,报告路径,创建时间
0,2026-04-01,k1,rows_count,42815.00,42816.000000,1.000000,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 02:51:27
1,2026-04-01,k1,rows_count,42815.00,42816.000000,1.000000,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 03:06:05
2,2026-04-01,k1,total_alpha,-15468.22,-15468.216830,0.003170,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 02:51:27
3,2026-04-01,k1,total_alpha,-15468.22,-15468.216830,0.003170,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 03:06:05
4,2026-04-01,k1,total_bundle_fee,5451.65,5451.651412,0.001412,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 02:51:27
5,2026-04-01,k1,total_bundle_fee,5451.65,5451.651412,0.001412,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 03:06:05
6,2026-04-01,k1,total_fee,5679.50,5679.505474,0.005474,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 02:51:27
7,2026-04-01,k1,total_fee,5679.50,5679.505474,0.005474,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 03:06:05
8,2026-04-01,k1,total_gas_fee,227.85,227.854062,0.004062,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 02:51:27
9,2026-04-01,k1,total_gas_fee,227.85,227.854062,0.004062,fail,/Users/yy/.openclaw/workspace/analysis/K1_2026...,2026-04-03 03:06:05


In [32]:
# 可选：关闭连接
# conn.close()

## 可视化看板增强

在表格基础上补充趋势和异常可视化，便于快速发现问题。

In [ ]:
# 可视化依赖
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Hiragino Sans GB', 'Heiti SC', 'Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# C) 看板趋势图（按日期）
if 'df_dashboard' in globals() and not df_dashboard.empty:
    d = df_dashboard.copy()
    d['日期'] = pd.to_datetime(d['日期'], errors='coerce')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    sns.lineplot(data=d, x='日期', y='今日总盈亏', hue='数据源', marker='o', ax=axes[0])
    axes[0].set_title('每日今日总盈亏趋势')
    axes[0].axhline(0, color='black', linewidth=0.8)

    sns.lineplot(data=d, x='日期', y='校验失败数', hue='数据源', marker='o', ax=axes[1])
    axes[1].set_title('每日校验失败数趋势')
    axes[1].axhline(0, color='black', linewidth=0.8)

    plt.tight_layout()
    plt.show()
else:
    print('请先运行“看板总览”单元。')

In [ ]:
# D) 规模区间热力图（alpha累计）
if 'df_size' in globals() and not df_size.empty:
    p = df_size.copy()
    p['日期'] = p['日期'].astype(str)
    p['规模区间'] = p['规模区间'].astype(str)
    mat = p.pivot_table(index='规模区间', columns='数据源', values='alpha累计', aggfunc='sum')

    plt.figure(figsize=(8, 4))
    sns.heatmap(mat, annot=True, fmt='.1f', cmap='RdYlGn', center=0)
    plt.title('规模区间 alpha累计热力图')
    plt.tight_layout()
    plt.show()
else:
    print('请先运行“规模区间”单元。')

In [ ]:
# E) 校验异常 Top 指标（失败频次）
if 'df_reconcile' in globals() and not df_reconcile.empty:
    fail_df = df_reconcile[df_reconcile['状态'] == 'fail'].copy()
    if not fail_df.empty:
        top_fail = fail_df['指标名'].value_counts().head(15).reset_index()
        top_fail.columns = ['指标名', '失败次数']

        plt.figure(figsize=(10, 4))
        sns.barplot(data=top_fail, x='指标名', y='失败次数')
        plt.title('校验失败指标 Top15')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print('当前筛选条件下没有 fail 异常。')
else:
    print('请先运行“报告校验异常”单元。')